# LAB: Visualiser, Tester et Déboguer avec LangGraph Studio
## Debugging & Visualization for LangChain Agents
### Master SDIA - Prof. RETAL SARA

Ce notebook couvre l'utilisation de LangGraph Studio pour visualiser, tester et déboguer des agents intelligents.

## 📊 Qu'est-ce que LangGraph Studio?

**LangGraph Studio** est un environnement interactif graphique qui permet de:

✨ **Visualization**
- 📈 Voir le flux du graphe (nodes, edges)
- 🔍 Inspecter les inputs/outputs de chaque étape
- 📊 Visualiser l'architecture de l'agent

🧪 **Testing**
- 💬 Envoyer des messages (HumanMessage)
- 🔄 Exécuter l'agent étape par étape
- ⏸️ Pause et inspection à chaque étape

🐛 **Debugging**
- 🔎 Vérifier les valeurs de chaque nœud
- 📝 Voir les logs détaillés
- 🛠️ Identifier les problèmes dans les outils
- 🚀 Tester RAG, agents, et workflows

### Cas d'usage:
- Développement d'agents complexes
- Debugging de comportements inattendus
- Monitoring en production
- Optimisation des workflows

## ⚙️ PARTIE 1: Créer un compte et obtenir une clé API

### Étape 1: Créer un compte LangSmith

In [ ]:
print("""
🚀 CRÉATION D'UN COMPTE LANGSMITH
="*70

Étape 1: Accédez au site officiel
  → https://smith.langchain.com/

Étape 2: Se connecter ou créer un compte
  □ Cliquez sur "Sign up" si vous n'avez pas de compte
  □ Ou "Log in" si vous en avez déjà un

Étape 3: Méthodes de connexion disponibles
  ✓ Google
  ✓ GitHub
  ✓ Email

Choisissez la méthode qui vous convient le mieux.
""")

print("\n💡 Conseil: Utilisez GitHub si vous développez avec LangChain")
print("   (Synchronisation plus facile avec vos projets)")

### Étape 2: Générer une clé API

In [ ]:
print("""
🔑 GÉNÉRATION D'UNE CLÉ API LANGSMITH
="*70

Une fois connecté:

1. Accédez à vos paramètres
   → Cliquez sur votre profil (bas gauche ou haut)
   → Allez dans "Settings"

2. Ouvrez la section API Keys
   → Dans le menu: Cliquez sur "API Keys"

3. Générez une nouvelle clé
   → Cliquez sur "Create API Key"
   → Donnez un nom descriptif (ex: "langgraph-project")
   → Cliquez sur "Create"

4. Copiez la clé
   → Vous verrez quelque chose comme:
      lsv2_pt_5ed0290dd44e42828db379e3eb88e474_4ec88ba
   → Gardez-la en sécurité!
""")

print("\n⚠️  IMPORTANT: Ne partagez JAMAIS votre clé API!")
print("   Elle est comme un mot de passe pour votre compte.")

### Étape 3: Ajouter la clé au projet

In [ ]:
print("""
📝 CONFIGURATION DE LA CLÉ API DANS LE PROJET
="*70

Créez un fichier .env à la racine du projet:

# .env
LANGSMITH_API_KEY=lsv2_pt_votre_clé_ici

Ensuite, en Python, chargez-la:

```python
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("LANGSMITH_API_KEY")
```

Ou définissez-la directement:

```python
import os
os.environ["LANGSMITH_API_KEY"] = "votre_clé_ici"
```
""")

print("\n✅ Votre clé est maintenant prête à être utilisée!")

---

## 🤖 PARTIE 2: Créer un Agent pour LangGraph Studio

### Structure d'un agent compatible avec LangGraph

In [ ]:
print("""
🏗️ STRUCTURE D'UN AGENT LANGGRAPH
="*70

Voici un exemple d'agent simple (agent_simple.py):

```python
from langchain_ollama import ChatOllama
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 1. Définir les tools
@tool
def rag_search(query: str) -> str:
    \"\"\"Recherche des informations dans le texte.\"\"\"
    results = "Le personnage principal est un jeune homme nommé Jack, qui découvre un ancien artefact magique."
    return results

# 2. Initialiser le modèle
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

# 3. Créer l'agent
agent = create_agent(
    model=llm,
    tools=[rag_search],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Tu es un assistant spécialisé dans l'analyse de texte. "
        "Utilise l'outil rag_search pour répondre aux questions. "
        "Réponds de manière précise et cite les informations du contexte."
    )
)

# 4. Important: exporter l'agent
# Cette ligne est nécessaire pour LangGraph Studio
# __all__ = ["agent"]
```

Points clés:
✓ Les tools doivent être décorés avec @tool
✓ Le modèle doit être configuré
✓ L'agent doit être créé avec create_agent()
✓ L'agent doit être la variable globale principale
""")

### Exemple complet d'un agent

In [ ]:
# Voici un exemple qu'on peut exécuter localement

from langchain.tools import tool
from langchain.messages import HumanMessage

print("✅ Création d'un agent exemple...\n")

# Définir un tool simple
@tool
def search_book(query: str) -> str:
    """Recherche des informations dans le livre."""
    # Base de données simulée
    book_data = {
        "personnage": "Le personnage principal est Jack, un jeune aventurier",
        "artefact": "Un ancien artefact magique découvert dans les ruines",
        "quete": "Jack part en quête pour comprendre les pouvoirs de l'artefact",
    }
    
    for key, value in book_data.items():
        if key.lower() in query.lower():
            return value
    
    return "Information non trouvée dans le livre."

print("Tool 'search_book' créé avec succès")
print(f"  Description: {search_book.description}")
print(f"  Arguments: {search_book.args}")

# Tester le tool
print("\n🧪 Test du tool:")
result = search_book.invoke("personnage")
print(f"  Query: 'personnage'")
print(f"  Résultat: {result}")

---

## 📋 PARTIE 3: Fichier de Configuration LangGraph

### Qu'est-ce que langgraph.json?

In [ ]:
print("""
📄 FICHIER DE CONFIGURATION: langgraph.json
="*70

Le fichier langgraph.json est un pont entre votre code Python
et le runtime LangGraph Studio/CLI.

Il indique au système:
  ✓ Où se trouvent vos agents
  ✓ Comment les charger
  ✓ Quel environnement utiliser
  ✓ Comment lancer le projet

Structure du fichier:
""")

config_example = """{
  "graphs": {
    "agent_simple": "./agent_simple.py:agent",
    "agent_rag": "./agents/rag_agent.py:rag_agent",
    "agent_sql": "./agents/sql_agent.py:sql_agent"
  },
  "env": "./.env",
  "source": {
    "kind": "uv",
    "root": "."
  }
}"""

print(config_example)

print("""

Explication des champs:

  "graphs": {
    "agent_simple": "./agent_simple.py:agent"
    |
    ├─ "agent_simple" = Nom de l'agent dans Studio
    ├─ "./agent_simple.py" = Chemin du fichier Python
    └─ ":agent" = Variable de l'agent dans le fichier
  }
  
  "env": "./.env"
    └─ Chemin du fichier .env (pour les API keys)
  
  "source": {
    "kind": "uv",     # Utilise UV pour gérer les dépendances
    "root": "."       # Racine du projet
  }
""")

### Créer le fichier langgraph.json

In [ ]:
import json
import os

print("\n📝 Création du fichier langgraph.json\n")

# Configuration pour un projet simple
langgraph_config = {
    "graphs": {
        "agent_simple": "./agent_simple.py:agent"
    },
    "env": "./.env",
    "source": {
        "kind": "uv",
        "root": "."
    }
}

# Afficher la configuration
print("Configuration:")
print(json.dumps(langgraph_config, indent=2))

print("\n✅ Sauvegardez ceci dans un fichier: langgraph.json")
print("   Emplacement: À la racine de votre projet")
print("\nContenu du fichier:")
print(json.dumps(langgraph_config, indent=2))

### Configuration pour plusieurs agents

In [ ]:
print("""
🔀 CONFIGURATION POUR PLUSIEURS AGENTS
="*70

Si vous avez plusieurs agents:

```json
{
  "graphs": {
    "simple_agent": "./agents/simple.py:agent",
    "rag_agent": "./agents/rag.py:rag_agent",
    "sql_agent": "./agents/sql.py:sql_agent",
    "mcp_agent": "./agents/mcp.py:mcp_agent",
    "chef_agent": "./agents/chef.py:chef"
  },
  "env": "./.env",
  "source": {
    "kind": "uv",
    "root": "."
  }
}
```

Dans LangGraph Studio, vous pourrez:
  • Basculer entre les agents via la dropdown
  • Tester chacun indépendamment
  • Comparer leurs comportements
  • Déboguer les interactions
""")

---

## 🚀 PARTIE 4: Installer et Lancer LangGraph Studio

### Installation du CLI LangGraph

In [ ]:
print("""
🛠️ INSTALLATION DE LANGGRAPH CLI
="*70

Étape 1: Installer le package LangGraph CLI

  Dans votre terminal (avec venv activé):

  $ uv add "langgraph-cli[inmem]"

  Cela installe:
  ✓ langgraph-cli: Le CLI pour lancer Studio
  ✓ [inmem]: Support pour la mémoire en mémoire

Étape 2: Vérifier l'installation

  $ langgraph --version

  Vous devriez voir la version du CLI
""")

print("\n💾 Configuration requise dans pyproject.toml:")
print("")
print("  langchain")
print("  langchain-ollama")
print("  langchain-core")
print("  langgraph")
print("  langgraph-cli[inmem]")
print("  python-dotenv")

### Lancer LangGraph Studio localement

In [ ]:
print("""
▶️ LANCER LANGGRAPH STUDIO
="*70

Commande pour démarrer:

  $ langgraph dev

Résultat attendu:

  Starting LangGraph Studio...
  
  📊 LangGraph Studio running at: http://localhost:8000
  
  Loading graphs from langgraph.json...
  ✓ Loaded: agent_simple
  ✓ Loaded: rag_agent
  ✓ Loaded: sql_agent

Étapes suivantes:

  1. Ouvrez votre navigateur
  2. Allez à: http://localhost:8000
  3. Sélectionnez un agent dans la dropdown
  4. Commencez à tester!

Pour arrêter:

  Pressez Ctrl+C dans le terminal
""")

print("\n⚠️  IMPORTANT: Assurez-vous que:")
print("  ✓ .env est créé et contient les API keys")
print("  ✓ langgraph.json est à la racine du projet")
print("  ✓ Les fichiers Python existent et sont valides")
print("  ✓ Ollama est en cours d'exécution (si vous l'utilisez)")

---

## 🎯 Utiliser LangGraph Studio

### Interface principale

In [ ]:
print("""
🖥️ INTERFACE DE LANGGRAPH STUDIO
="*70

L'interface contient:

┌─────────────────────────────────────────────────────────┐
│ HAUT:                                                   │
│  • Studio / agent_rag (Dropdown)  [Graph] [Chat]       │
│  • Connected (Status)                                  │
├──────────────────┬──────────────────────────────────────┤
│ GAUCHE:          │ CENTRE:                              │
│ • Home           │ • Visualisation du graphe            │
│ • Tracing        │   ┌─────────┐                        │
│ • Monitoring     │   │ __start__│                       │
│ • Studio         │   └────┬────┘                        │
│ • Deployments    │        │                             │
│                  │   ┌────▼─────┐                       │
│ (Settings)       │   │  model   │                       │
│                  │   └────┬─────┘                       │
│                  │        │                             │
│                  │  ┌─────┴─────┐                       │
│                  │  │           │                       │
│                  │  ▼           ▼                       │
│                  │ tools    __end__                     │
├──────────────────┼──────────────────────────────────────┤
│ DROITE (Input):                                        │
│ Messages:                                              │
│ [+ Message] (Add message)                              │
│                                                        │
│ [Submit] (Run the agent)                               │
└─────────────────────────────────────────────────────────┘

Fonctionnalités:

1️⃣ Graph Tab:
   • Visualise l'architecture de l'agent
   • Montre le flux des données
   • Affiche les nodes et edges

2️⃣ Chat Tab:
   • Interface de chat comme ChatGPT
   • Envoyer des messages simplement
   • Voir les réponses en temps réel

3️⃣ Memory & Interrupts:
   • Gérer la mémoire de l'agent
   • Mettre en pause à certains points
   • Reprendre l'exécution

4️⃣ Panels latéraux:
   • Home: Vue d'ensemble
   • Tracing: Historique détaillé
   • Monitoring: Performance
   • Deployments: Publier l'agent
""")

### Tester un agent

In [ ]:
print("""
🧪 TESTER UN AGENT DANS LANGGRAPH STUDIO
="*70

Étapes pour tester:

1. Sélectionner l'agent
   └─ Dropdown "agent_simple" en haut

2. Basculer sur l'onglet "Graph"
   └─ Pour voir l'architecture

3. Aller à la section "Input"
   └─ À droite de l'écran

4. Ajouter un message
   └─ Cliquez sur "+ Message"
   └─ Entrez votre question
   └─ Ex: "Who is the main character?"

5. Cliquer sur "Submit"
   └─ L'agent exécute étape par étape

6. Observer l'exécution
   └─ Chaque node s'exécute en ordre
   └─ Les arrêtes s'illuminent
   └─ Les données traversent le graphe

7. Inspecter les données
   └─ Cliquez sur un node
   └─ Voyez l'input et output
   └─ Déboguer si nécessaire

Outils de debugging disponibles:

  🔍 Voir le contenu exact de chaque étape
  📊 Vérifier les entrées/sorties
  🐛 Identifier où l'agent va mal
  ⏸️ Mettre en pause et inspecter
  🔄 Reprendre l'exécution
""")

### Déboguer un problème

In [ ]:
print("""
🐛 DÉBOGUER UN AGENT QUI NE FONCTIONNE PAS
="*70

Scénario 1: L'agent ignore les tools
  ❌ L'agent répond sans utiliser les tools
  
  Solution:
  1. Vérifier que les tools sont bien passés à create_agent()
  2. Vérifier que les tools ont une description claire
  3. Vérifier que le system_prompt encourage l'utilisation des tools
  4. Relancer avec `langgraph dev`

Scénario 2: Un tool retourne une erreur
  ❌ Tool execution failed: [Erreur]
  
  Solution:
  1. Cliquer sur le node du tool
  2. Voir le message d'erreur exact
  3. Vérifier les paramètres en input
  4. Corriger le code du tool
  5. Relancer

Scénario 3: L'agent entre dans une boucle infinie
  ❌ L'agent ne s'arrête pas
  
  Solution:
  1. Cliquer sur "Interrupts" à droite
  2. Choisir "Interrupt before" sur un node
  3. Relancer le test
  4. L'agent s'arrêtera avant ce node
  5. Vérifier où il boucle

Scénario 4: L'agent n'utilise pas la mémoire
  ❌ L'agent oublie les messages précédents
  
  Solution:
  1. Cliquer sur "Memory" à droite
  2. Vérifier que le thread_id est le même
  3. Vérifier que checkpointer est configuré
  4. Vérifier que messages sont sauvegardés

Utiliser l'onglet "Tracing":
  • Voir l'historique complet de l'exécution
  • Identifier exactement où ça s'est mal passé
  • Retracer les appels aux tools
  • Analyser les tokens utilisés
""")

---

## 📚 Structure d'un Projet Complet

In [ ]:
print("""
📁 STRUCTURE RECOMMANDÉE D'UN PROJET LANGGRAPH
="*70

mon-projet/
│
├── langgraph.json              # Configuration principale
├── .env                        # Variables d'environnement
├── pyproject.toml              # Dépendances (uv)
├── uv.lock                     # Lock file
│
├── agents/
│   ├── simple_agent.py         # Agent simple
│   ├── rag_agent.py            # Agent RAG
│   ├── sql_agent.py            # Agent SQL
│   └── __init__.py
│
├── tools/
│   ├── search_tools.py         # Tools de recherche
│   ├── database_tools.py        # Tools de BD
│   └── __init__.py
│
├── resources/
│   ├── mcp_local_server.py     # Serveur MCP (optionnel)
│   └── data.json               # Données (optionnel)
│
└── tests/
    ├── test_agents.py          # Tests unitaires
    └── __init__.py

Contenu de .env:

  LANGSMITH_API_KEY=lsv2_pt_...
  OPENAI_API_KEY=sk-...
  GROQ_API_KEY=gsk_...
  OLLAMA_MODEL=llama3.2:3b

Contenu de langgraph.json:

  {
    "graphs": {
      "simple": "./agents/simple_agent.py:agent",
      "rag": "./agents/rag_agent.py:rag_agent",
      "sql": "./agents/sql_agent.py:sql_agent"
    },
    "env": "./.env",
    "source": {
      "kind": "uv",
      "root": "."
    }
  }

Démarrage:

  $ uv venv
  $ source .venv/bin/activate
  $ uv add langchain langgraph "langgraph-cli[inmem]"
  $ langgraph dev
  
  → http://localhost:8000
""")

---

## 🎯 Bonnes Pratiques avec LangGraph Studio

In [ ]:
print("""
✨ BONNES PRATIQUES
="*70

1. Organisation du code
   ✓ Séparer les tools dans des fichiers
   ✓ Un agent par fichier
   ✓ Utiliser des dossiers logiques (agents/, tools/)
   ✓ Documenter les tools avec des docstrings

2. Configuration
   ✓ Toujours utiliser .env pour les secrets
   ✓ Ne JAMAIS hardcoder les API keys
   ✓ Vérifier que langgraph.json est valide
   ✓ Tester la configuration avec `langgraph dev`

3. Debugging
   ✓ Utiliser le Graph tab pour voir l'architecture
   ✓ Tester les tools individuellement d'abord
   ✓ Ajouter des logs détaillés dans les tools
   ✓ Utiliser les Interrupts pour pause à des points clés
   ✓ Consulter l'onglet Tracing pour les détails

4. Testing
   ✓ Créer des test cases simples d'abord
   ✓ Augmenter la complexité progressivement
   ✓ Tester les cas limites (empty input, errors)
   ✓ Vérifier la performance (tokens, temps)

5. Monitoring
   ✓ Garder un oeil sur la section Monitoring
   ✓ Vérifier la performance et les erreurs
   ✓ Analyser les patterns d'utilisation
   ✓ Optimiser les tools lents

6. Déploiement
   ✓ Tester complètement localement d'abord
   ✓ Utiliser la section Deployments pour publier
   ✓ Monitorer en production
   ✓ Garder les versions de l'agent
""")

---

## 🔗 Intégration avec les Labs Précédents

In [ ]:
print("""
🔀 UTILISER LANGGRAPH STUDIO AVEC LES AUTRES LABS
="*70

Lab 1 - Prompt Engineering:
  • Tester différents prompts dans Studio
  • Voir comment le modèle répond
  • Optimiser via l'interface graphique

Lab 2 - Agents LangChain:
  • Visualiser l'architecture de l'agent
  • Tester les tools développés
  • Déboguer les interactions tool-agent
  • Améliorer les prompts système

Lab 3 - RAG & SQL:
  • Tester le RAG avec des requêtes variées
  • Vérifier les embeddings et recherches
  • Tester les requêtes SQL générées
  • Déboguer les problèmes de mémoire

Lab 4 - MCP:
  • Tester les serveurs MCP local/distant
  • Vérifier la récupération dynamique des tools
  • Déboguer les connexions MCP
  • Visualiser les multi-serveurs

Workflow complet:

  1. Développer l'agent localement
     └─ Fichier agent_*.py

  2. Ajouter à langgraph.json
     └─ "my_agent": "./agents/my_agent.py:agent"

  3. Lancer `langgraph dev`
     └─ Interface graphique démarrée

  4. Tester dans Studio
     └─ Via Graph tab ou Chat tab

  5. Déboguer les problèmes
     └─ Utiliser Tracing et Interrupts

  6. Optimiser et redéployer
     └─ Relancer `langgraph dev`

  7. Publier en production
     └─ Onglet Deployments
""")

---

## 📝 Résumé et Prochaines Étapes

In [ ]:
print("""
✅ RÉSUMÉ DU LAB LANGGRAPH STUDIO
="*70

Ce que vous avez appris:

1. Créer et configurer un compte LangSmith
   ✓ Générer une clé API
   ✓ Ajouter la clé au projet

2. Créer des agents LangGraph-compatibles
   ✓ Structure basique d'un agent
   ✓ Définir des tools
   ✓ Configuration du modèle

3. Configurer le projet avec langgraph.json
   ✓ Format du fichier
   ✓ Ajouter plusieurs agents
   ✓ Configuration de l'environnement

4. Installer et lancer LangGraph Studio
   ✓ Installer langgraph-cli
   ✓ Lancer le serveur local
   ✓ Accéder à http://localhost:8000

5. Utiliser l'interface Studio
   ✓ Tester les agents
   ✓ Visualiser l'architecture
   ✓ Déboguer les problèmes
   ✓ Analyser les performances

Prochaines étapes:

  1. Intégrer vos agents créés dans les labs 1-4
  2. Lancer `langgraph dev` sur chacun
  3. Tester à travers l'interface Studio
  4. Déboguer et optimiser
  5. Déployer en production (via Deployments)

Resources utiles:

  • LangSmith: https://smith.langchain.com/
  • LangGraph Docs: https://langchain-ai.github.io/langgraph/
  • LangChain Docs: https://python.langchain.com/
  • Community: https://discord.gg/langchain
""")

print("\n" + "="*70)
print("✨ Vous êtes prêt à utiliser LangGraph Studio!")
print("="*70)